# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [7]:
from pathlib import Path
import numpy as np
import pandas as pd

# Locate the repository root from the notebook's working directory.
repo_root = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "data/raw/content_refresh_anonymized.csv").exists()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Could not find data/raw/content_refresh_anonymized.csv from this notebook's folder.")

df = pd.read_csv(repo_root / "data/raw/content_refresh_anonymized.csv")

# This starter target is a current-trend proxy, not a later future outcome.
y = df["trend_direction"].eq("down").astype("int8").rename("is_declining_proxy")
context = df[["content_id", "client_id"]].copy()  # Grouping/review only; never part of X.

# These source measurements come from the 30 days before the latest 30-day outcome window.
X_raw = pd.DataFrame(index=df.index)
for source_column in ["impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"]:
    X_raw[f"log1p_{source_column}"] = np.log1p(df[source_column].clip(lower=0))

# A previous-window CTR is undefined when that window had no impressions.
previous_impressions = df["impressions_prev_30d"].where(df["impressions_prev_30d"] > 0)
X_raw["ctr_prev_30d"] = df["clicks_prev_30d"].div(previous_impressions) * 100
X_raw["content_type"] = df["content_type"]

numeric_features = [column for column in X_raw if column != "content_type"]
categorical_features = ["content_type"]

# Keep identifiers separately. The model only receives X_raw after train-fitted preprocessing.
assert len(X_raw) == len(y) == len(context)
assert set(X_raw.columns).isdisjoint({"content_id", "client_id", "trend_direction", "trend_pct"})
assert X_raw["content_type"].notna().any()

print(f"Rows: {len(X_raw):,}; raw candidate features: {X_raw.shape[1]}")
print(f"Proxy positive rate: {y.mean():.1%}")
print(f"Numeric candidates: {numeric_features}")
print(f"Categorical candidates: {categorical_features}")
print(f"Rows with missing prior-window CTR (no prior impressions): {X_raw['ctr_prev_30d'].isna().sum():,}")
print("No imputation or encoding has been fitted yet; that happens using training rows only.")

Rows: 30,000; raw candidate features: 5
Proxy positive rate: 54.2%
Numeric candidates: ['log1p_impressions_prev_30d', 'log1p_clicks_prev_30d', 'log1p_sessions_prev_30d', 'ctr_prev_30d']
Categorical candidates: ['content_type']
Rows with missing prior-window CTR (no prior impressions): 3,388
No imputation or encoding has been fitted yet; that happens using training rows only.


## 2. Feature notes (meaning, missing, categorical, available-when?)

**What this section asks:** explain each candidate feature in plain language, how blank values are handled, and whether it would exist when the prediction is made.

**Prediction-time assumption for this exercise:** pretend the decision is made at the start of the latest 30-day comparison window. The previous-30-day counts and content type are then available. The target is still a current-trend proxy, not a true future outcome; the warehouse capstone must define a later target window.

**Why the next cell exists:** it makes a compact feature dictionary from the actual `X_raw` columns. Missing CTR stays missing here; the model pipeline will learn its imputation value from training rows only and add a missingness indicator.

In [8]:
feature_meanings = {
    "log1p_impressions_prev_30d": "log(1 + Google search impressions in the previous 30-day window)",
    "log1p_clicks_prev_30d": "log(1 + Google search clicks in the previous 30-day window)",
    "log1p_sessions_prev_30d": "log(1 + analytics sessions in the previous 30-day window)",
    "ctr_prev_30d": "clicks / impressions * 100 in the previous window; undefined if impressions are zero",
    "content_type": "structured content category; categorical, not a numeric rank",
}

feature_notes = pd.DataFrame([
    {
        "feature": column,
        "meaning": feature_meanings[column],
        "kind": "categorical" if column in categorical_features else "numeric",
        "missing_fraction": X_raw[column].isna().mean(),
        "available_at_assumed_decision_time": "yes, previous window / existing metadata",
        "planned_handling": (
            "most-frequent imputation + one-hot encoding, fitted on training clients"
            if column in categorical_features
            else "median imputation + missingness indicator + scaling, fitted on training clients"
        ),
    }
    for column in X_raw.columns
])

print("Feature dictionary (missing_fraction is a fraction from 0 to 1):")
feature_notes

Feature dictionary (missing_fraction is a fraction from 0 to 1):


,feature,meaning,kind,missing_fraction,available_at_assumed_decision_time,planned_handling
0,log1p_impressions_prev_30d,log(1 + Google search impressions in the previ...,numeric,0.000000,"yes, previous window / existing metadata",median imputation + missingness indicator + sc...
1,log1p_clicks_prev_30d,log(1 + Google search clicks in the previous 3...,numeric,0.000000,"yes, previous window / existing metadata",median imputation + missingness indicator + sc...
2,log1p_sessions_prev_30d,log(1 + analytics sessions in the previous 30-...,numeric,0.000000,"yes, previous window / existing metadata",median imputation + missingness indicator + sc...
3,ctr_prev_30d,clicks / impressions * 100 in the previous win...,numeric,0.112933,"yes, previous window / existing metadata",median imputation + missingness indicator + sc...
4,content_type,"structured content category; categorical, not ...",categorical,0.000000,"yes, previous window / existing metadata","most-frequent imputation + one-hot encoding, f..."


## 3. The leakage hunt

**What this section asks:** try to break your own feature set. Check for label-derived fields, measurements from the outcome period, product decisions, and validation that lets the same client appear on both sides.

**Timeline:**

```text
previous 30 days: candidate numeric features  |  decision point  |  latest 30 days: trend proxy outcome
```

The starter proxy is not a true future label: `trend_direction` is calculated from recent-versus-previous impressions. We can still test whether the *candidate inputs* avoid using its direct source. For validation, hold out whole clients. Then deliberately add `trend_direction` to a second model. That model is invalid by design; a much stronger score demonstrates why the field must be excluded.

**Why the next cell exists:** it fits both versions on the same client-group split and reports ROC AUC, average precision, Precision@50, and the test base rate. Preprocessing is fitted inside each training-only pipeline.

In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Keep every page from one client entirely in train OR test.
group_split = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_index, test_index = next(group_split.split(X_raw, y, groups=context["client_id"]))
X_train, X_test = X_raw.iloc[train_index], X_raw.iloc[test_index]
y_train, y_test = y.iloc[train_index], y.iloc[test_index]
train_clients = set(context.iloc[train_index]["client_id"])
test_clients = set(context.iloc[test_index]["client_id"])
assert train_clients.isdisjoint(test_clients), "A client appears in both train and test."

print(f"Training rows: {len(train_index):,} from {len(train_clients)} clients")
print(f"Test rows: {len(test_index):,} from {len(test_clients)} different clients")
print(f"Test positive rate (base rate): {y_test.mean():.1%}")


def make_model(categorical_columns):
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scale", StandardScaler()),
    ])
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("one_hot", OneHotEncoder(handle_unknown="ignore")),
    ])
    preprocessing = ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_columns),
    ])
    return Pipeline([
        ("preprocess", preprocessing),
        ("model", LogisticRegression(max_iter=1000, random_state=42)),
    ])


def score_model(model, feature_frame):
    model.fit(feature_frame.iloc[train_index], y_train)
    probabilities = model.predict_proba(feature_frame.iloc[test_index])[:, 1]
    top_k_positions = np.argsort(-probabilities)[:50]
    return {
        "ROC AUC": roc_auc_score(y_test, probabilities),
        "Average precision": average_precision_score(y_test, probabilities),
        "Precision@50": y_test.iloc[top_k_positions].mean(),
    }


# Valid comparison: only past-window candidates and metadata.
safe_model = make_model(categorical_features)
safe_scores = score_model(safe_model, X_raw)

# Deliberately invalid: trend_direction directly defines the proxy target.
X_with_leak = X_raw.assign(trend_direction=df["trend_direction"])
leaky_model = make_model(categorical_features + ["trend_direction"])
leaky_scores = score_model(leaky_model, X_with_leak)

comparison = pd.DataFrame(
    [safe_scores, leaky_scores],
    index=["Candidate features only", "INVALID: includes trend_direction"],
)
print("\nSame grouped split, same metrics:")
print(comparison.round(3).to_string())
print("\nInterpretation: the leaky row is a demonstration only. Since trend_direction creates y, its score must never be reported as model performance.")

Training rows: 22,885 from 24 clients
Test rows: 7,115 from 8 different clients
Test positive rate (base rate): 51.7%

Same grouped split, same metrics:
                                   ROC AUC  Average precision  Precision@50
Candidate features only              0.636              0.613           0.7
INVALID: includes trend_direction    1.000              1.000           1.0

Interpretation: the leaky row is a demonstration only. Since trend_direction creates y, its score must never be reported as model performance.


## 4. What I excluded and why

**What this section asks:** name the fields that the model refused to learn from and give a reason for each group. Exclusion is not the same as saying a field is useless; it means it is unsafe or outside this first, deliberately small feature set.

- **IDs:** `content_id` and `client_id` identify/group rows. The codes have no predictive meaning; `client_id` is used only to make the grouped split.
- **Label-derived fields:** `trend_direction` defines the proxy; `trend_pct` and `impressions_last_30d` are used to form/describe that same trend. They leak the answer.
- **Overlapping/current-window fields:** 90-day totals, most-recent 30-day counts, current-window rates/position, and tiers derived from those values include the outcome period. They are not prior-only inputs.
- **Generation metadata:** `provider_used` and `model_used` are documented as not model features for this starter task.
- **Other unselected fields:** omitted from this first compact baseline, not proven useless. They can be reconsidered after the decision-time contract and signal audit support them.
- **Product decisions:** flags/scores such as `health_score` are not present in this starter file; if encountered elsewhere, they are baselines to compare against, not ordinary features.

**Why the next cell exists:** it inventories the omitted source columns with a reason, confirms the listed buckets cover every field not used to build `X_raw`, and checks that IDs and direct label sources are absent from the valid feature frame.

In [10]:
model_input_sources = {
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d", "content_type",
}
excluded_groups = {
    "IDs: grouping only": {"content_id", "client_id"},
    "Label-derived / answer fields": {"trend_direction", "trend_pct", "impressions_last_30d"},
    "Current or overlapping windows": {
        "clicks_last_30d", "sessions_last_30d",
        "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
        "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
        "days_with_impressions", "days_with_sessions",
        "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
        "impression_tier", "position_tier",
    },
    "Generation metadata: not selected as model inputs": {"provider_used", "model_used"},
}

classified_exclusions = set().union(*excluded_groups.values())
remaining_fields = set(df.columns) - model_input_sources - classified_exclusions
excluded_groups["Other fields not selected for this compact baseline"] = remaining_fields

all_excluded = set().union(*excluded_groups.values())
assert all_excluded == set(df.columns) - model_input_sources, "Some unused source fields lack a reason."
feature_columns = list(X_raw.columns)
allowed_engineered_columns = [
    column in df.columns or column.startswith("log1p_") or column == "ctr_prev_30d"
    for column in feature_columns
]
assert all(allowed_engineered_columns), "Unexpected feature column."
assert set(X_raw.columns).isdisjoint({"content_id", "client_id", "trend_direction", "trend_pct", "impressions_last_30d"})

# Product-decision fields are not in this starter schema. If added in another dataset, treat them as baselines.
product_decision_fields = {"health_score", "needs_ctr_fix", "is_quick_win", "priority_score", "action_type"}
print(f"Product decision fields present in starter CSV: {sorted(product_decision_fields.intersection(df.columns))}")

exclusion_notes = pd.DataFrame([
    {"field": field, "reason": reason}
    for reason, fields in excluded_groups.items()
    for field in sorted(fields)
])
print(f"\nExcluded / unselected source fields: {len(exclusion_notes)}")
print(exclusion_notes.to_string(index=False))

Product decision fields present in starter CSV: []

Excluded / unselected source fields: 40
                 field                                              reason
             client_id                                  IDs: grouping only
            content_id                                  IDs: grouping only
  impressions_last_30d                       Label-derived / answer fields
       trend_direction                       Label-derived / answer fields
             trend_pct                       Label-derived / answer fields
       ai_sessions_90d                      Current or overlapping windows
        ai_traffic_pct                      Current or overlapping windows
          avg_position                      Current or overlapping windows
            clicks_90d                      Current or overlapping windows
       clicks_last_30d                      Current or overlapping windows
                   ctr                      Current or overlapping windows
 days_wi

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section is filled — feature construction, feature notes, leakage test, and exclusions are documented with checks.
- [x] The notebook runs top to bottom with no errors (verified in this workspace).
- [x] No client names, URLs, private queries, or row-level IDs are printed.
- [x] The deliberately leaky `trend_direction` result is labeled invalid and is not presented as model performance.
- [x] Claims describe an observed starter-data proxy; this is not a future prediction or causal result.
- [ ] Committed to my repo under `work/notebooks/` and submitted my repo URL on the assignment card. (Do this after reviewing the notebook.)